# H3 — Session-Open HL Gate vs P&L (W3)

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('hl_mod', '../notebooks/11_half_life_bars.py')
hl_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hl_mod)
hl_series_minutes = hl_mod.hl_series_minutes


In [ ]:
wm = WINDOWS_META['W3']
rk = wm['result_key']
ts = pd.read_parquet(RESULTS_DIR / rk / 'none' / 'timeseries.parquet')
ts.index = pd.to_datetime(ts.index)
trades = pd.read_parquet(RESULTS_DIR / rk / 'none' / 'trades.parquet')
trades = trades.sort_values('entry_time').reset_index(drop=True)

day_hl, day_cum_pnl = [], []
for i, d in enumerate(wm['days']):
    sub = ts[ts.index.date.astype(str) == d]
    # first 30 min of RTH
    rth_start = sub.index.min() + pd.Timedelta(minutes=0)
    thirty_min_mask = (sub.index >= rth_start) & (sub.index < rth_start + pd.Timedelta(minutes=30))
    open_dev = sub[thirty_min_mask]['dev'].dropna().values
    if len(open_dev) < 5:
        day_hl.append(np.nan); day_cum_pnl.append(np.nan); continue
    try:
        hl = hl_series_minutes(open_dev, bar_res=1, n_bars=len(open_dev), bar_secs=1)
        day_hl.append(np.nanmedian(hl))
    except Exception:
        day_hl.append(np.nan)
    # cumulative PnL through end of this day
    day_trades = trades[trades['entry_time'].dt.date.astype(str) == d]
    day_cum_pnl.append(day_trades['gross_usd'].sum())

fig, ax1 = plt.subplots(figsize=(12, 5))
x = range(len(wm['days']))
color_hl = '#3498db'
color_pnl = '#2ecc71'
bars = ax1.bar(x, day_hl, color=color_hl, alpha=0.6, label='Session-open HL (min)')
for i, (b, h) in enumerate(zip(bars, day_hl)):
    if h is not None and not np.isnan(h) and h > 120:
        b.set_edgecolor('red'); b.set_linewidth(2)
ax1.set_ylabel('OU Half-Life (minutes)', color=color_hl)
ax1.tick_params(axis='y', labelcolor=color_hl)
ax1.set_xticks(list(x)); ax1.set_xticklabels(wm['day_labels'])
ax1.axhline(120, color='red', ls='--', lw=0.8, label='HL=120 min threshold')

ax2 = ax1.twinx()
cum_pnl_cum = np.cumsum([v if v is not None and not np.isnan(v) else 0 for v in day_cum_pnl])
ax2.step(x, cum_pnl_cum, where='post', color=color_pnl, lw=2, label='Cumulative P&L')
ax2.axhline(0, color='black', lw=0.5)
ax2.set_ylabel('Cumulative P&L ($)', color=color_pnl)
ax2.tick_params(axis='y', labelcolor=color_pnl)

lines1, lbls1 = ax1.get_legend_handles_labels()
lines2, lbls2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, lbls1+lbls2, loc='upper left')
ax1.set_title('W3: Session-Open HL Gate vs Daily P&L\n(red border = HL > 120 min, spread not mean-reverting at open)')
save_fig(fig, 'H3_session_open_hl_gate_w3.png')
